# Section 04: 机器翻译（Seq2Seq）核心总结

## 任务定义
**翻译** = Seq2Seq（序列到序列）任务，输入一种语言的句子，输出另一种语言。

## 本节任务
用 `Helsinki-NLP/opus-mt-en-fr`（基于 MarianMT 的 en→fr 翻译模型）在 **KDE4** 数据集上微调。

## Seq2Seq 架构
```
                    ┌──────────────────────────────────┐
英文句子             │  Encoder (编码源语言)              │
"Hello world"  →   │  [CLS] Hello world [EOS]  →  记忆  │
                    └──────────────────────────────────┘
                                    ↓ 交叉注意力
                    ┌──────────────────────────────────┐
法文句子             │  Decoder (自回归生成目标语言)       │
"Bonjour monde"←   │  [BOS] Bonjour monde [EOS]        │
                    └──────────────────────────────────┘
```

## 完整流程
```
KDE4 数据集（英文+法文对）
    ↓ preprocess_function（源语言+目标语言分别 tokenize）
    ↓ DataCollatorForSeq2Seq（自动构造 decoder_input_ids）
    ↓ AutoModelForSeq2SeqLM
    ↓ Seq2SeqTrainer（predict_with_generate=True）
    ↓ 评估指标：BLEU
```

---
## 第一步：数据集准备

In [3]:
from datasets import load_dataset

# KDE4 是软件本地化数据集，包含英文-法文平行句子对
raw_datasets = load_dataset("ArmelRandy/kde4")

# 只有 train split，手动切分 validation
split_datasets = raw_datasets["train"].train_test_split(train_size=0.9, seed=20)
split_datasets["validation"] = split_datasets.pop("test")  # 重命名

# 每个样本是一个翻译对
print(split_datasets["train"][1])
# {'en': 'Default to expanded threads', 'fr': 'Par défaut, développer les fils de discussion'}

{'en': 'The PERMUT() function returns the number of permutations. The first parameter is the number of elements, and the second parameter is the number of elements used in the permutation.', 'fr': "La fonction PERMUT() renvoie le nombre de permutations. Le premier paramètre est le nombre d'éléments et le second est le nombre d'éléments à permuter."}


In [4]:
# 用已有模型快速验证翻译质量（baseline）
# transformers 5.x 移除了 pipeline("translation") 任务，改为直接使用模型
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"

_tok = AutoTokenizer.from_pretrained(model_checkpoint)
_mdl = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

def translator(texts):
    inputs = _tok(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = _mdl.generate(**inputs)
    return [{"translation_text": t} for t in _tok.batch_decode(outputs, skip_special_tokens=True)]

print(translator("Default to expanded threads"))
# [{'translation_text': 'Par défaut pour les threads élargis'}]  ← 微调前效果一般

/home/work/miniforge3/envs/llm-hf/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[{'translation_text': 'Par défaut pour les threads élargis'}]


---
## 第二步（关键）：双语 Tokenize — text_target

翻译任务的 tokenize 比单任务复杂：源语言和目标语言需要**分别 tokenize**。

为什么？MarianMT 的 encoder/decoder 各自有不同的词表和特殊处理逻辑。

> ⚠️ **transformers 5.x 变更**：`as_target_tokenizer()` 上下文管理器已移除，
> 改用 `tokenizer(text_target=...)` 参数传入目标语言文本。

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

en_sentence = "Default to expanded threads"
fr_sentence = "Par défaut, développer les fils de discussion"

# 源语言 tokenize（encoder 输入）
inputs = tokenizer(en_sentence)

# 目标语言 tokenize（decoder 目标）— transformers 5.x 用 text_target 参数
targets = tokenizer(text_target=fr_sentence)

print("英文 tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"]))
print("法文 tokens:", tokenizer.convert_ids_to_tokens(targets["input_ids"]))

英文 tokens: ['▁Default', '▁to', '▁expanded', '▁thread', 's', '</s>']
法文 tokens: ['▁Par', '▁défaut', ',', '▁développer', '▁les', '▁fils', '▁de', '▁discussion', '</s>']


In [7]:
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    """
    批量处理翻译对：
    - 源语言（英文）→ input_ids
    - 目标语言（法文）→ labels（供计算 loss）
    transformers 5.x：用 text_target 参数替代 as_target_tokenizer()
    """
    # 同时 tokenize 源语言和目标语言，labels 自动放入 model_inputs
    model_inputs = tokenizer(
        examples['en'], text_target=examples['fr'],
        max_length=max_input_length, truncation=True,
    )
    return model_inputs

tokenized_datasets = split_datasets.map(
    preprocess_function, batched=True,
    remove_columns=split_datasets["train"].column_names,
)

Map:   0%|          | 0/18052 [00:00<?, ? examples/s]

Map:   0%|          | 0/2006 [00:00<?, ? examples/s]

---
## 第三步：DataCollatorForSeq2Seq — 自动构造 decoder_input_ids

Seq2Seq 训练时，decoder 的输入是目标序列**右移一位**（Teacher Forcing）：
```
labels:           [A, B, C, EOS]   ← 训练目标（模型要预测的）
decoder_input_ids:[BOS, A, B, C]   ← decoder 的实际输入（每步用上一个真实词）
```
`DataCollatorForSeq2Seq` 自动完成这个右移操作，同时对变长序列做 padding。

In [8]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# model 参数用于让 collator 知道 pad_token_id（decoder padding 用）
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 查看 batch 的 key：多了 decoder_input_ids！
batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
print("Batch keys:", list(batch.keys()))
print("labels:", batch["labels"])
print("decoder_input_ids:", batch["decoder_input_ids"])
# labels 中用 -100 padding（不计 loss），decoder_input_ids 用 pad_token_id

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Batch keys: ['input_ids', 'attention_mask', 'labels', 'decoder_input_ids']
labels: tensor([[   80,   952, 10460,   559,  7075,   401,    28, 19730,    19,   384,
             5,   329, 18888,  2252,     3,    60,   698, 23408,    43,    19,
           384,    20,     6, 13635,    11,    19,   787,    43,    19,   384,
            20,     6, 13635,    17,   329, 18888,   108,     3,     0,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100],
        [  104,   167,    15,   871,    38,  6502,   614,    14,     6,  2569,
            22,   767,  8810,     5,     8,  6442,   359,  6199, 13415,    27,
           301,   331,   402, 38492,   301,   548,   344, 12754, 11979,   153,
           402, 29033,   954,   529,   309, 17770,   301,   602,   402, 29033,
           416,    23,   863,     5,  3745,     3,     0]])
decoder_input_ids: tensor([[59513,    80,   952, 10460,   559,  7075,   401,    28, 19730,    19,
           384,     5,   329, 18888,  2252,     3,    60,   698, 234

---
## 第四步：评估指标 — BLEU

**BLEU (Bilingual Evaluation Understudy)**：翻译任务的标准评估指标
- 计算预测译文与参考译文之间的 n-gram 重叠度
- 范围 0~100，越高越好
- 考虑 1-gram、2-gram、3-gram、4-gram 的精确率，再用简洁惩罚因子调整

**BLEU 的局限性**：
- 只看词语重叠，不考虑语义
- 对翻译顺序敏感
- 多个参考译文时更准确

In [9]:
import evaluate
import numpy as np

metric = evaluate.load("sacrebleu")  # sacrebleu 是 BLEU 的标准实现

# 演示 BLEU 计算
predictions = ["This plugin lets you translate web pages between several languages automatically."]
references  = [["This plugin allows you to automatically translate web pages between several languages."]]
print(metric.compute(predictions=predictions, references=references))
# score 约 46.75

{'score': 46.750469682990165, 'counts': [11, 6, 4, 3], 'totals': [12, 11, 10, 9], 'precisions': [91.66666666666667, 54.54545454545455, 40.0, 33.333333333333336], 'bp': 0.9200444146293233, 'sys_len': 12, 'ref_len': 13}


In [10]:
def compute_metrics(eval_preds):
    """
    Seq2Seq 模型输出是 token id 序列，需要 decode 成文本再计算 BLEU。
    注意：labels 中 -100 需要替换为 pad_token_id 才能 decode。
    """
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # -100 无法 decode，先替换为 pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]  # sacrebleu 要求列表的列表

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

---
## 第五步：使用 Seq2SeqTrainer 训练

`Seq2SeqTrainer` 继承自 `Trainer`，关键额外参数：
- `predict_with_generate=True`：评估时用 `model.generate()` 生成序列（而不是取 argmax），更准确

In [11]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers.utils.notebook import NotebookProgressCallback

args = Seq2SeqTrainingArguments(
    output_dir="marian-finetuned-kde4-en-to-fr",
    eval_strategy="no",                # 只在最后评估（翻译评估耗时）
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,        # 关键：用 generate() 而不是 argmax
    fp16=True,
    push_to_hub=True,
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,        # transformers 5.x 用 processing_class 替代 tokenizer
    compute_metrics=compute_metrics,
)

# 训练前 BLEU: 39.27 → 训练后 BLEU: 52.94
trainer.remove_callback(NotebookProgressCallback)
trainer.evaluate(max_length=max_target_length)
trainer.add_callback(NotebookProgressCallback)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
500,1.010697
1000,0.893828
1500,0.851380


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1695, training_loss=0.9088765282194875, metrics={'train_runtime': 109.4979, 'train_samples_per_second': 494.585, 'train_steps_per_second': 15.48, 'total_flos': 1758417299177472.0, 'train_loss': 0.9088765282194875, 'epoch': 3.0})

In [ ]:
trainer.remove_callback(NotebookProgressCallback)
trainer.evaluate(max_length=max_target_length)

---
## 第六步：Accelerate 自定义循环的关键差异

Seq2Seq 的 Accelerate 循环与 Token Classification 类似，但评估时有重要区别：
不能用 `outputs.logits.argmax()`，必须用 `model.generate()` 生成序列。

In [ ]:
# Accelerate 评估阶段的关键代码（训练阶段与 section-02 完全相同）
import torch

# model.eval()
# for batch in eval_dataloader:
#     with torch.no_grad():
#         # ⚠️ 关键：用 generate() 而不是 forward()
#         # 必须先 unwrap_model，因为 generate() 不被 DDP 包装支持
#         generated_tokens = accelerator.unwrap_model(model).generate(
#             batch["input_ids"],
#             attention_mask=batch["attention_mask"],
#             max_length=128,
#         )
#
#     # 多 GPU 时，不同 batch 生成序列长度不一，需要 pad 后 gather
#     generated_tokens = accelerator.pad_across_processes(
#         generated_tokens, dim=1, pad_index=tokenizer.pad_token_id
#     )
#     labels = accelerator.pad_across_processes(batch["labels"], dim=1, pad_index=-100)
#
#     predictions_gathered = accelerator.gather(generated_tokens)
#     labels_gathered = accelerator.gather(labels)

print("核心差异：")
print("  Token Classification: outputs.logits.argmax()")
print("  Seq2Seq Translation:  accelerator.unwrap_model(model).generate(...)")

---
## 总结

### 核心知识点速查

| 概念 | 说明 |
|------|------|
| Seq2Seq 架构 | Encoder 读取源语言 → Decoder 生成目标语言，通过交叉注意力连接 |
| `text_target` 参数 | transformers 5.x 替代 `as_target_tokenizer()`，目标语言 tokenize 方式 |
| Teacher Forcing | 训练时 decoder 输入是真实目标序列（右移），而非上一步预测结果 |
| `DataCollatorForSeq2Seq` | 自动构造 `decoder_input_ids`（labels 右移+BOS），并 pad |
| `predict_with_generate=True` | 评估用 autoregressive generate()，比 argmax 更准确 |
| `processing_class` | transformers 5.x 替代 `Trainer(tokenizer=...)`  |
| BLEU 分数 | 越高越好；微调前 39.27 → 微调后 52.94（+13 BLEU）|

### transformers 5.x 主要 API 变更
| 旧 API | 新 API |
|--------|--------|
| `pipeline("translation", model=...)` | 直接用 `AutoModelForSeq2SeqLM` + `AutoTokenizer` |
| `with tokenizer.as_target_tokenizer():` | `tokenizer(text_target=...)` |
| `Trainer(tokenizer=tokenizer)` | `Trainer(processing_class=tokenizer)` |
| `evaluation_strategy` | `eval_strategy` |

### 与 Token Classification 的对比
```
Token Classification：
  - 模型：AutoModelForTokenClassification
  - 输出：每个 token 的分类 logit
  - 评估：logits.argmax()

Seq2Seq Translation：
  - 模型：AutoModelForSeq2SeqLM
  - 输出：自回归生成的 token 序列
  - 评估：model.generate()（beam search 等策略）
```